#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Text Classification - Document Classification (Inglés)

In [1]:
#Instalar Librerías
!pip install datasets scikit-learn pandas -q
!pip install --upgrade transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.3 MB/s eta 0:00:00


In [2]:
#Importar Librerías
import pandas as pd
from datasets import Dataset
import torch

In [3]:
# Load dataset
df = pd.read_csv("pubmed_multilabel_classification.csv")
df.columns = ["Title", "Psych", "Phenomena", "Tech", "Health"]

In [4]:
#Lista de Labels
df["labels"] = df[["Psych", "Phenomena", "Tech", "Health"]].values.tolist()
df = df[["Title", "labels"]]
df.loc[:, "labels"] = df["labels"].apply(lambda x: [float(i) for i in x])

In [7]:
#Definir Labels & Hugging Face Dataset
label_names = ["Psych", "Phenomena", "Tech", "Health"]
num_labels = len(label_names)
dataset = Dataset.from_pandas(df)

In [8]:
#Importar Librerías
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

In [9]:
#Crear Tokenizador
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
#Función de Tokenización
def tokenize(batch):
  return tokenizer(batch["Title"], padding=True, truncation=True)

In [11]:
#Tokenización & Formateo Labels
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [13]:
#Train & Test Split
dataset = dataset.train_test_split(test_size=0.2)

In [14]:
#Creación de Modelo
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type='multi_label_classification'
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
#Importar Librerías
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

In [16]:
#Desahiblitar Weights & Biases
import os
os.environ["WANDB_DISABLED"] = "true"

In [17]:
#Configuración Entrenamiento
args = TrainingArguments(
    output_dir="pubmed-multilabel",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    report_to="none"
)

In [18]:
#Función de Evaluación
def compute_metrics(pred):
    logits, labels = pred
    preds = (logits > 0).astype(int)
    f1_weighted = f1_score(labels, preds, average="weighted")
    f1_per_class = f1_score(labels, preds, average=None)
    return {"Weighted F1": f1_weighted, "Per-Class F1": f1_per_class}

In [19]:
#Importar Librerías
from transformers import DataCollatorWithPadding

In [20]:
#Agregar DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [21]:
#Parametrizar Entrenamiento
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [22]:
#Entrenar Modelo
trainer.train()

Step,Training Loss


TrainOutput(global_step=450, training_loss=0.3904705132378472, metrics={'train_runtime': 75.2327, 'train_samples_per_second': 47.852, 'train_steps_per_second': 5.981, 'total_flos': 142452527817600.0, 'train_loss': 0.3904705132378472, 'epoch': 3.0})

In [23]:
#Evaluar Modelo
results = trainer.evaluate()

In [24]:
#Revisar Resultados
print("Weighted F1:", results['eval_Weighted F1'])
print("Per-Class F1:", results['eval_Per-Class F1'])

Weighted F1: 0.7582651055561062
Per-Class F1: [0.63157895 0.82640587 0.5        0.75261324]


In [25]:
#Utilizar el Modelo en Modo Inferencia
new_article = "Overview of the multiview and 3D extensions of high efficiency video coding"

# Tokenizar el texto
inputs = tokenizer(new_article, return_tensors="pt", truncation=True, padding=True)

In [27]:
#Mover Inputs a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {key: value.to(device) for key, value in inputs.items()}

In [29]:
# Usar el modelo en modo inferencia
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = logits.sigmoid()
    prediction = (probs > 0.5).int()

In [30]:
#Revisar Resultados
print("Predicción Binaria:", prediction.tolist())
print("Probabilidades (Continuas):", probs.tolist())

Predicción Binaria: [[0, 1, 0, 1]]
Probabilidades (Continuas): [[0.1557934582233429, 0.6961225867271423, 0.2558354437351227, 0.7828962206840515]]
